# คู่มือการอัด Dataset (แบบ LSTM-learn + Siamese)

การอัดแบบนี้เหมือนไฟล์ `LSTM-learn` คือ **อัดต่อเนื่องอัตโนมัติทีละหลายคลิป** และเก็บ **ทีละเฟรมเป็นโฟลเดอร์**
แต่แยก dataset เป็น 2 ชุด คือ `correct` (ท่าถูก) และ `incorrect` (ท่าผิด) แล้วนำไปเทรนด้วย **Siamese Network** เหมือนเดิม

### โครงสร้างโฟลเดอร์ที่ได้
```
dataset/
├── correct/          ← ท่าวันทยหัตถ์ที่ถูกต้อง
│   ├── 0/  0.npy ... 59.npy   (1 คลิป = 6 วินาที = 60 เฟรม, เฟรมละ 218 ค่า)
│   ├── 1/
│   └── ...
└── incorrect/        ← ท่าที่ผิด
    ├── 0/
    └── ...
```

### จุด (joint) ที่ใช้ detect — MediaPipe Holistic
| ส่วน | จุดที่ใช้ | ค่าที่เก็บ |
|---|---|---|
| หน้า | Pose 0–10 (จมูก ตา หู ปาก) | 11 × 4 |
| ไหล่ | Pose 11, 12 | 2 × 4 |
| ศอก | Pose 13, 14 | 2 × 4 |
| ข้อมือ | Pose 15, 16 | 2 × 4 |
| ขา (สะโพก, เข่า) | Pose 23–26 | 4 × 4 |
| ข้อเท้า | Pose 27, 28 | 2 × 4 |
| มือซ้าย + มือขวา | Hand 21 จุด/ข้าง (นิ้วครบ) | 2 × 21 × 3 |
| **รวม** | | **218 ค่า/เฟรม** |

- Pose เก็บ (x, y, z, visibility), มือเก็บ (x, y, z) ทุกจุดถูก normalize เทียบกับกึ่งกลางไหล่และความกว้างไหล่
- ถ้าตรวจไม่เจอมือข้างไหน จะใส่ 0 แทน แต่ถ้าไม่เจอตัว (pose) คลิปจะเริ่มใหม่
- ไฟล์ที่อัดด้วยแบบเก่า (132 ค่า) จะถูกข้ามอัตโนมัติ ต้องอัดใหม่ทั้งหมด

### ขั้นตอนการอัด
1. ไปที่ cell **1. Collect Dataset** ตั้งค่า `CLASS_TO_RECORD = "correct"` แล้วกด Run
2. หน้าจอจะขึ้น **GET READY** นับถอยหลัง → ยืนเตรียมท่าเริ่มต้น
3. พอขึ้น **REC ●** ให้ทำท่าวันทยหัตถ์ **1 ครั้งให้จบภายใน 6 วินาที** (ระบบเก็บ 10 เฟรม/วินาที = 60 เฟรม)
   - แนะนำจังหวะ: ยืนตรง (~1 วิ) → ยกมือทำความเคารพ → ค้างไว้ (~2–3 วิ) → ลดมือลง → ยืนตรง
   - ใช้จังหวะเดียวกันทุกคลิป ทั้งท่าถูกและท่าผิด
4. ระบบจะบันทึกคลิปแล้วพักให้อัตโนมัติ แล้ววนอัดคลิปถัดไปจนครบ `NO_SEQUENCES`
5. เปลี่ยนเป็น `CLASS_TO_RECORD = "incorrect"` แล้ว Run อีกครั้ง เพื่ออัดท่าผิด
6. รัน cell **1.1** เพื่อตรวจจำนวนคลิป / ลบคลิปที่อัดพลาด

> Run ซ้ำได้เรื่อย ๆ ระบบจะ **ต่อเลขโฟลเดอร์ถัดไปให้เอง ไม่เขียนทับของเดิม**

### ปุ่มลัดระหว่างอัด
| ปุ่ม | การทำงาน |
|---|---|
| `Q` | หยุดอัดทั้งหมด (คลิปที่บันทึกไปแล้วยังอยู่) |
| `S` | ข้ามคลิปนี้ (ไม่บันทึก) แล้วเริ่มนับถอยหลังใหม่ |

### จำนวนที่แนะนำ
| ชุด | ขั้นต่ำ | แนะนำ |
|---|---|---|
| correct | 20 คลิป | 40–60 คลิป |
| incorrect | 20 คลิป | 40–60 คลิป |

ต้องมี **อย่างน้อย 14 คลิปต่อชุด** เพราะข้อมูลจะแบ่ง train/val/test = 70/15/15 และแต่ละส่วนต้องมีอย่างน้อย 2 คลิป

### เทคนิคให้ได้ dataset ที่ดี
**ท่าถูก (correct)**
- อัดหลายคนถ้าทำได้ (รูปร่าง/ความสูงต่างกัน)
- ขยับระยะห่างจากกล้องบ้าง (ใกล้-ไกล), ยืนเอียงซ้าย-ขวาเล็กน้อย
- ความเร็วในการยกมือ ช้า-ปกติ-เร็ว ปนกัน

**ท่าผิด (incorrect)** — ให้ครอบคลุมความผิดที่เจอจริง เช่น
- ใช้มือผิดข้าง (มือซ้าย)
- ปลายนิ้วไม่แตะขอบหมวก/คิ้ว, ยกมือต่ำหรือสูงเกินไป
- ศอกไม่กาง หรือกางมากเกินไป
- ฝ่ามือหงาย / นิ้วไม่ชิด / ข้อมืองอ
- ตัวเอียง ไม่อยู่ในท่าตรง
- ยืนเฉย ๆ ไม่ทำท่า หรือทำท่าอื่น
- กระจายจำนวนแต่ละแบบให้ใกล้เคียงกัน

**ทั่วไป**
- ต้อง **เห็นทั้งตัวตั้งแต่หัวถึงเท้า** (เพราะใช้ข้อเท้าและขา) และเห็นมือชัด แสงสว่างพอ ไม่ย้อนแสง
- ดูให้มีเส้นโครงมือขึ้นบนจอระหว่างอัด ถ้ามือไม่ขึ้นบ่อยให้ขยับเข้าใกล้กล้องหรือเพิ่มแสง
- ถ้า MediaPipe จับตัวไม่ได้ระหว่างอัด ระบบจะ **เริ่มคลิปนั้นใหม่อัตโนมัติ** (ไม่บันทึกคลิปที่ขาดเฟรม)
- ตำแหน่งกล้อง/การกลับภาพ (`flip`) ต้องเหมือนกันทั้งตอนอัดและตอนใช้งานจริง

### Label ท่าผิด (สำหรับ feedback)
- **ตอนอัดใหม่:** ตั้ง `CLASS_TO_RECORD = "incorrect"` และ `ERROR_TYPE = "all"` ใน cell 1 → ระบบจะพาอัดครบทั้ง 10 ประเภท โดยขึ้น **หน้าแจ้งท่าภาษาไทย** ก่อนเริ่มแต่ละประเภท (กด SPACE เพื่อเริ่ม) และติด label ให้เอง
  - อัดประเภทเดียว: `ERROR_TYPE = "hand_low"` / เลือกหลายประเภท: `ERROR_TYPE = ["hand_low", "body_tilted"]`
- **คลิปที่อัดไปแล้ว:** รัน cell 1.2 เพื่อดูโครงกระดูกแต่ละคลิปแล้วกดเลข 1–9 และ 0 เลือกความผิด
- label เก็บที่ `dataset/incorrect/labels.json` (ไม่กระทบการเทรน Siamese)
- แนะนำให้แต่ละประเภทมี **อย่างน้อย 5 คลิป**

### หลังอัดเสร็จ
รัน cell ต่อไปตามลำดับ: **2. สร้างคู่ (pairs)** → **3. Build Siamese** → **4. Evaluate** → **4.5 Calibrate feedback** → **5. Test** (จะขึ้น feedback ภาษาไทยบอกว่าผิดตรงไหน)

# 1. Collect Dataset (อัดต่อเนื่องแบบ LSTM-learn)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
import time
import json
from PIL import Image, ImageDraw, ImageFont

# -------------------------
# 1. Configuration
# -------------------------
CLASS_TO_RECORD = "incorrect"   # "correct" = ท่าถูก, "incorrect" = ท่าผิด
NO_SEQUENCES = 6              # จำนวนคลิป (ท่าผิด = จำนวนคลิปต่อ 1 ประเภท)
CLIP_SECONDS = 6              # ความยาวแต่ละคลิป (วินาที)
SAMPLE_FPS = 10               # เก็บ 10 เฟรมต่อวินาที
SEQUENCE_LENGTH = CLIP_SECONDS * SAMPLE_FPS   # = 60 เฟรมต่อคลิป
BREAK_SECONDS = 2             # เวลาพักเตรียมท่าก่อนอัดแต่ละคลิป
CAMERA_ID = 0

# ท่าผิดที่จะอัด (ใช้เฉพาะ CLASS_TO_RECORD = "incorrect")
#   "all"                    = อัดครบทุกประเภทตามลำดับ ประเภทละ NO_SEQUENCES คลิป
#   "hand_low"               = อัดประเภทเดียว
#   ["hand_low", "body_tilted"] = อัดเฉพาะที่เลือก ตามลำดับ
#   None                     = ไม่ใส่ label (label ภายหลังได้ใน cell 1.2)
ERROR_TYPE = "all"

DATA_DIR = Path("dataset")
LABELS_PATH = DATA_DIR / "incorrect" / "labels.json"

# รายการความผิด: code → (ชื่อภาษาไทย, วิธีทำท่าผิดตอนอัด)
ERROR_INFO = {
    "wrong_hand": (
        "มือผิดข้าง",
        "ทำวันทยหัตถ์ด้วย มือซ้าย แทนมือขวา"
    ),
    "hand_low": (
        "ปลายนิ้วต่ำกว่าคิ้ว",
        "ยกมือทำความเคารพ แต่ให้ปลายนิ้ว อยู่ระดับตา หรือแก้ม"
    ),
    "hand_high": (
        "ปลายนิ้วสูงกว่าคิ้ว",
        "ยกมือทำความเคารพ แต่ให้ปลายนิ้ว อยู่เหนือหน้าผาก"
    ),
    "elbow_at_shoulder": (
        "ศอกอยู่ระดับเดียวกับไหล่",
        "ยกมือทำความเคารพ แต่ยกศอกสูง จนอยู่ระดับเดียวกับไหล่"
    ),
    "body_tilted": (
        "ตัวเอียง",
        "ทำท่าวันทยหัตถ์ แต่เอียงตัว หรือเอียงไหล่"
    ),
    "heels_apart": (
        "ส้นเท้าไม่ชิด",
        "ทำท่าวันทยหัตถ์ แต่ยืนแยกส้นเท้า ออกจากกัน"
    ),
    "no_salute": (
        "ไม่ทำท่า",
        "ยืนตรงเฉย ๆ หรือทำท่าอื่น ไม่ยกมือทำความเคารพ"
    ),
    "no_head_turn": (
        "ไม่สะบัดหน้าไปทางซ้าย",
        "ยกอกอึ้บ ทำท่าครบ แต่หน้ามองตรง ไม่สะบัดหน้าไปทางซ้าย"
    ),
    "wrong_head_turn": (
        "สะบัดหน้าผิดไปทางขวา",
        "ทำท่าครบ แต่สะบัดหน้าไป ทางขวา แทนทางซ้าย"
    ),
    "left_arm_not_at_side": (
        "มือซ้ายไม่แนบลำตัว",
        "ทำท่าวันทยหัตถ์ แต่มือซ้าย กางออก ไม่แนบข้างลำตัว"
    ),
}
ERROR_TYPES = list(ERROR_INFO)

# แผนการอัด: 1 รายการ = 1 คลิป (None = ไม่มี label)
if CLASS_TO_RECORD == "correct" or ERROR_TYPE is None:
    plan = [None] * NO_SEQUENCES
else:
    if ERROR_TYPE == "all":
        record_types = ERROR_TYPES
    elif isinstance(ERROR_TYPE, str):
        record_types = [ERROR_TYPE]
    else:
        record_types = list(ERROR_TYPE)

    unknown = [e for e in record_types if e not in ERROR_TYPES]

    if unknown:
        raise ValueError(f"ไม่รู้จัก ERROR_TYPE {unknown} ต้องเป็น {ERROR_TYPES}")

    plan = [e for e in record_types for _ in range(NO_SEQUENCES)]

print(f"แผนการอัด ({CLASS_TO_RECORD}): {len(plan)} คลิป")

for error in dict.fromkeys(plan):
    name = "ท่าถูกต้อง" if CLASS_TO_RECORD == "correct" else (
        ERROR_INFO[error][0] if error else "ไม่ระบุประเภท"
    )
    print(f"  - {name}: {plan.count(error)} คลิป")

CLASSES = ["correct", "incorrect"]

if CLASS_TO_RECORD not in CLASSES:
    raise ValueError(f"CLASS_TO_RECORD ต้องเป็น {CLASSES}")

for class_name in CLASSES:
    (DATA_DIR / class_name).mkdir(
        parents=True,
        exist_ok=True
    )

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

WINDOW_NAME = "SaluteAI - Data Collection"


# -------------------------
# 2. Extract 218 features
# -------------------------
# Pose joints ที่ใช้ (index ของ MediaPipe Pose)
POSE_JOINTS = {
    "face": list(range(0, 11)),   # จมูก ตา หู ปาก (11 จุด)
    "shoulders": [11, 12],
    "elbows": [13, 14],
    "wrists": [15, 16],
    "hips": [23, 24],             # สะโพก (ต้นขา)
    "knees": [25, 26],            # เข่า
    "ankles": [27, 28],           # ข้อเท้า
}
POSE_INDEXES = [i for group in POSE_JOINTS.values() for i in group]

HAND_POINTS = 21                  # จุดบนมือข้างละ 21 จุด (นิ้วครบ)

# 23 pose x (x, y, z, visibility) + 2 มือ x 21 x (x, y, z) = 92 + 126 = 218
FEATURES = len(POSE_INDEXES) * 4 + 2 * HAND_POINTS * 3


def extract_landmarks(results):
    if results.pose_landmarks is None:
        return None

    landmarks = results.pose_landmarks.landmark

    # Shoulder landmarks
    left_shoulder = landmarks[11]
    right_shoulder = landmarks[12]

    center_x = (
        left_shoulder.x + right_shoulder.x
    ) / 2

    center_y = (
        left_shoulder.y + right_shoulder.y
    ) / 2

    scale = np.sqrt(
        (left_shoulder.x - right_shoulder.x) ** 2
        + (left_shoulder.y - right_shoulder.y) ** 2
    )

    scale = max(scale, 1e-6)

    values = []

    # Pose: หน้า ไหล่ ศอก ข้อมือ สะโพก เข่า ข้อเท้า
    for index in POSE_INDEXES:
        landmark = landmarks[index]

        # Normalize coordinates relative to shoulders
        x = (landmark.x - center_x) / scale
        y = (landmark.y - center_y) / scale
        z = landmark.z / scale
        visibility = landmark.visibility

        values.extend([x, y, z, visibility])

    # Hands: ถ้าตรวจไม่เจอมือข้างไหน ใส่ 0 แทน (เหมือน LSTM-learn)
    for hand in [
        results.left_hand_landmarks,
        results.right_hand_landmarks
    ]:
        if hand is None:
            values.extend([0.0] * HAND_POINTS * 3)
            continue

        for landmark in hand.landmark:
            x = (landmark.x - center_x) / scale
            y = (landmark.y - center_y) / scale
            z = landmark.z / scale

            values.extend([x, y, z])

    return np.array(values, dtype=np.float32)


def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_holistic.POSE_CONNECTIONS
    )
    mp_drawing.draw_landmarks(
        image,
        results.left_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS
    )
    mp_drawing.draw_landmarks(
        image,
        results.right_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS
    )


# -------------------------
# 3. Folder helpers (เหมือน MP_Data ใน LSTM-learn)
# -------------------------
def count_sequences(class_name):
    return len([
        d for d in (DATA_DIR / class_name).iterdir()
        if d.is_dir() and d.name.isdigit()
    ])


def next_sequence_index(class_name):
    # ต่อเลขโฟลเดอร์จากของเดิม ไม่เขียนทับ
    indexes = [
        int(d.name) for d in (DATA_DIR / class_name).iterdir()
        if d.is_dir() and d.name.isdigit()
    ]

    return max(indexes) + 1 if indexes else 0


def save_label(sequence_index, errors):
    labels = {}

    if LABELS_PATH.exists():
        labels = json.loads(LABELS_PATH.read_text(encoding="utf-8"))

    labels[str(sequence_index)] = list(errors)

    LABELS_PATH.write_text(
        json.dumps(labels, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )


def save_sequence(class_name, sequence_index, frames):
    sequence_dir = DATA_DIR / class_name / str(sequence_index)
    sequence_dir.mkdir(parents=True, exist_ok=True)

    for frame_num, keypoints in enumerate(frames):
        np.save(sequence_dir / f"{frame_num}.npy", keypoints)

    return sequence_dir


def put_text(image, text, org, font, scale, color, thickness=2, line_type=cv2.LINE_AA):
    # ตัวหนังสือหนา มีขอบดำ อ่านชัดทุกพื้นหลัง
    thickness += 1
    cv2.putText(image, text, org, font, scale, (0, 0, 0), thickness + 4, line_type)
    cv2.putText(image, text, org, font, scale, color, thickness, line_type)


# -------------------------
# Thai text (cv2 เขียนภาษาไทยไม่ได้ ใช้ Pillow + ฟอนต์ไทยแทน)
# -------------------------
THAI_FONT_CANDIDATES = [
    Path("../../KanitFont/Kanit-Bold.ttf"),
    Path("C:/Windows/Fonts/Kanit-Medium.ttf"),
    Path("C:/Windows/Fonts/tahomabd.ttf"),
    Path("C:/Windows/Fonts/LeelawUI.ttf"),
]
_thai_fonts = {}


def thai_font(size):
    if size not in _thai_fonts:
        path = next((p for p in THAI_FONT_CANDIDATES if p.exists()), None)

        if path is None:
            raise FileNotFoundError("ไม่พบฟอนต์ภาษาไทย")

        _thai_fonts[size] = ImageFont.truetype(str(path), size)

    return _thai_fonts[size]


def wrap_thai(draw, text, font, max_width):
    # ตัดบรรทัดตรงช่องว่าง เมื่อข้อความยาวเกินจอ
    lines, line = [], ""

    for word in text.split(" "):
        candidate = f"{line} {word}".strip()

        if line and draw.textlength(candidate, font=font) > max_width:
            lines.append(line)
            line = word
        else:
            line = candidate

    lines.append(line)
    return lines


def draw_thai(image, items):
    # items: [(text, (x, y), size, (B, G, R)), ...] ตัวหนังสือขอบดำ หนา อ่านชัด
    pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)

    for text, (x, y), size, color in items:
        font = thai_font(size)

        for line in wrap_thai(draw, text, font, image.shape[1] - x - 15):
            draw.text(
                (x, y), line, font=font,
                fill=tuple(color[::-1]),
                stroke_width=max(2, size // 10),
                stroke_fill=(0, 0, 0)
            )
            y += int(size * 1.35)

    image[:] = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


def draw_header(image, text, color):
    cv2.rectangle(image, (0, 0), (image.shape[1], 40), color, -1)
    put_text(
        image, text, (10, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7, (255, 255, 255), 2, cv2.LINE_AA
    )


def resample_sequence(times, frames):
    # เลือกเฟรมที่ใกล้เวลาเป้าหมายที่สุด ให้ได้ SEQUENCE_LENGTH เฟรมเท่ากันทุกคลิป
    # ไม่ว่ากล้อง + MediaPipe จะได้กี่ FPS
    times = np.asarray(times, dtype=np.float64)
    times = times - times[0]
    targets = np.linspace(0.0, times[-1], SEQUENCE_LENGTH)
    indexes = np.abs(times[None, :] - targets[:, None]).argmin(axis=1)

    return [frames[i] for i in indexes]


def read_frame(cap, pose):
    success, frame = cap.read()

    if not success:
        return None, None

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    draw_landmarks(frame, results)

    return frame, results


def task_text(error):
    if CLASS_TO_RECORD == "correct":
        return "ท่าถูกต้อง", "ทำท่าวันทยหัตถ์ ให้ถูกต้องครบทุกจุด"

    if error is None:
        return "ท่าผิด (ไม่ระบุประเภท)", "ทำท่าผิดแบบใดก็ได้"

    return ERROR_INFO[error]


def darken(image, alpha, top=0, bottom=None):
    bottom = image.shape[0] if bottom is None else bottom
    region = image[top:bottom]
    image[top:bottom] = (region * (1 - alpha)).astype(np.uint8)


def draw_task_panel(image, error):
    # แถบบอกท่าที่ต้องทำ ระหว่าง GET READY และ REC
    title, detail = task_text(error)
    prefix = "ท่าถูก" if CLASS_TO_RECORD == "correct" else "ท่าผิด"

    darken(image, 0.6, 42, 130)
    draw_thai(image, [
        (f"{prefix}: {title}", (12, 46), 30, (255, 255, 255)),
        (detail, (12, 88), 20, (200, 255, 200)),
    ])


def show_intro(cap, pose, error, type_number, type_total, clips):
    # หน้าแจ้งท่าที่จะอัด รอกด SPACE ก่อนเริ่ม
    title, detail = task_text(error)

    while True:
        frame, results = read_frame(cap, pose)

        if frame is None:
            continue

        darken(frame, 0.65)

        items = [
            (f"ท่าที่ {type_number}/{type_total}", (25, 25), 26, (200, 200, 200)),
            (title, (25, 70), 44, (255, 255, 255)),
            (detail, (25, 140), 28, (200, 255, 200)),
            (f"อัด {clips} คลิป  คลิปละ {CLIP_SECONDS} วินาที", (25, 260), 24, (255, 255, 255)),
        ]

        if error is not None:
            items.append(
                ("ส่วนอื่นทำให้ถูกต้องทั้งหมด ผิดแค่จุดนี้จุดเดียว", (25, 300), 22, (255, 255, 255))
            )

        items.append(
            ("กด SPACE เพื่อเริ่ม     กด Q เพื่อออก", (25, frame.shape[0] - 55), 26, (255, 255, 255))
        )

        draw_thai(frame, items)
        cv2.imshow(WINDOW_NAME, frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord(" "):
            return True

        if key == ord("q"):
            return False


# -------------------------
# 4. Record loop
# -------------------------
cap = cv2.VideoCapture(CAMERA_ID)

if not cap.isOpened():
    raise RuntimeError(
        "เปิดกล้องไม่ได้ กรุณาตรวจสอบ CAMERA_ID"
    )

start_folder = next_sequence_index(CLASS_TO_RECORD)
saved_count = 0
stop_all = False

print(f"Recording '{CLASS_TO_RECORD}' starting at folder {start_folder}")

try:
    with mp_holistic.Holistic(
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as pose:

        clip_number = 0
        previous_error = "__start__"
        plan_types = list(dict.fromkeys(plan))

        while clip_number < len(plan) and not stop_all:
            sequence_index = start_folder + saved_count
            current_error = plan[clip_number]

            # --- หน้าแจ้งท่าใหม่ (ทุกครั้งที่เปลี่ยนประเภท) ---
            if current_error != previous_error:
                previous_error = current_error

                if not show_intro(
                    cap, pose, current_error,
                    plan_types.index(current_error) + 1,
                    len(plan_types),
                    plan.count(current_error)
                ):
                    break

            # --- พักเตรียมท่า (GET READY) ---
            break_start = time.time()

            while time.time() - break_start < BREAK_SECONDS:
                frame, results = read_frame(cap, pose)

                if frame is None:
                    continue

                remaining = BREAK_SECONDS - (time.time() - break_start)

                draw_header(
                    frame,
                    f"{CLASS_TO_RECORD.upper()}  clip {clip_number + 1}/{len(plan)}"
                    f"  (folder {sequence_index})",
                    (245, 117, 16)
                )

                put_text(
                    frame,
                    f"GET READY {remaining:.1f}",
                    (120, 220),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5, (255, 255, 255), 4, cv2.LINE_AA
                )

                if results.pose_landmarks is None:
                    put_text(
                        frame,
                        "Body not detected - stand in frame",
                        (10, 160),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0, 0, 255), 2
                    )

                put_text(
                    frame,
                    "Q: Quit",
                    (10, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 255, 255), 2
                )

                draw_task_panel(frame, current_error)
                cv2.imshow(WINDOW_NAME, frame)

                if cv2.waitKey(1) & 0xFF == ord("q"):
                    stop_all = True
                    break

            if stop_all:
                break

            # --- อัด CLIP_SECONDS วินาที ---
            frames = []
            times = []
            skipped = False
            record_start = time.time()

            while time.time() - record_start < CLIP_SECONDS:
                frame, results = read_frame(cap, pose)

                if frame is None:
                    continue

                features = extract_landmarks(results)

                if features is None:
                    # ตัวหลุดเฟรม → เริ่มคลิปนี้ใหม่ เพื่อไม่ให้มีเฟรมขาดช่วง
                    frames.clear()
                    times.clear()
                    record_start = time.time()

                    put_text(
                        frame,
                        "Body lost - restarting clip",
                        (10, 160),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7, (0, 0, 255), 2
                    )
                else:
                    frames.append(features)
                    times.append(time.time())

                elapsed = time.time() - record_start

                draw_header(
                    frame,
                    f"REC  {CLASS_TO_RECORD.upper()}  clip {clip_number + 1}/{len(plan)}",
                    (0, 0, 200)
                )

                # Progress bar (ตามเวลา)
                progress = int(
                    frame.shape[1] * min(elapsed / CLIP_SECONDS, 1.0)
                )
                cv2.rectangle(
                    frame,
                    (0, frame.shape[0] - 10),
                    (progress, frame.shape[0]),
                    (0, 255, 0), -1
                )

                put_text(
                    frame,
                    f"Time: {elapsed:.1f}/{CLIP_SECONDS}s   S: Skip  Q: Quit",
                    (10, frame.shape[0] - 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 255, 255), 2
                )

                draw_task_panel(frame, current_error)
                cv2.imshow(WINDOW_NAME, frame)

                key = cv2.waitKey(1) & 0xFF

                if key == ord("q"):
                    stop_all = True
                    break

                if key == ord("s"):
                    skipped = True
                    break

            if stop_all:
                break

            if skipped:
                print(f"Skipped clip {clip_number + 1} - recording again")
                continue

            raw_fps = len(frames) / CLIP_SECONDS

            if len(frames) < SEQUENCE_LENGTH // 2:
                print(
                    f"FPS ต่ำเกินไป ({raw_fps:.1f} FPS) - อัดคลิปนี้ใหม่"
                )
                continue

            frames = resample_sequence(times, frames)

            saved_dir = save_sequence(
                CLASS_TO_RECORD,
                sequence_index,
                frames
            )

            saved_count += 1
            clip_number += 1

            if current_error:
                save_label(sequence_index, [current_error])

            print(
                f"Saved: {saved_dir}  ({len(frames)} frames, camera {raw_fps:.1f} FPS)"
                + (f"  label={current_error}" if current_error else "")
            )

finally:
    cap.release()
    cv2.destroyAllWindows()

print("\nSaved this session:", saved_count)

for class_name in CLASSES:
    print(f"Total {class_name}: {count_sequences(class_name)} clips")

## 1.1 ตรวจสอบจำนวนคลิป / ลบคลิปที่อัดพลาด

In [ ]:
from pathlib import Path
import shutil
import json
import numpy as np

DATA_DIR = Path("dataset")
SEQUENCE_LENGTH = 60   # 6 วินาที x 10 FPS
FEATURES = 218   # 23 pose joints x 4 + 2 hands x 21 x 3

# ใส่เลขโฟลเดอร์ที่ต้องการลบ เช่น {"correct": [3, 7], "incorrect": [0]}
# ปล่อยว่างไว้ถ้าแค่ต้องการตรวจสอบ
DELETE = {"correct": [], "incorrect": []}

for class_name, indexes in DELETE.items():
    for index in indexes:
        target = DATA_DIR / class_name / str(index)

        if target.is_dir():
            shutil.rmtree(target)
            print("Deleted:", target)

            # ลบ label ของคลิปนั้นด้วย
            labels_path = DATA_DIR / "incorrect" / "labels.json"

            if class_name == "incorrect" and labels_path.exists():
                labels = json.loads(labels_path.read_text(encoding="utf-8"))

                if labels.pop(str(index), None) is not None:
                    labels_path.write_text(
                        json.dumps(labels, indent=2, ensure_ascii=False),
                        encoding="utf-8"
                    )
        else:
            print("Not found:", target)

for class_name in ["correct", "incorrect"]:
    class_dir = DATA_DIR / class_name
    folders = sorted(
        (d for d in class_dir.iterdir() if d.is_dir() and d.name.isdigit()),
        key=lambda d: int(d.name)
    )

    broken = []

    for folder in folders:
        files = [folder / f"{i}.npy" for i in range(SEQUENCE_LENGTH)]

        if not all(f.exists() for f in files):
            broken.append(folder.name)
            continue

        if np.load(files[0]).shape != (FEATURES,):
            broken.append(folder.name)

    print(f"{class_name}: {len(folders)} clips", end="")
    print(f"  | broken: {broken}" if broken else "  | all OK")

## 1.2 Label ท่าผิด (สำหรับ feedback logic)
เล่นซ้ำโครงกระดูกของแต่ละคลิปใน `dataset/incorrect/` แล้วกดปุ่มเลือกว่าผิดอะไร → บันทึกลง `dataset/incorrect/labels.json`

| ปุ่ม | การทำงาน |
|---|---|
| `1`–`9`, `0` | เลือก / ยกเลิก ประเภทความผิด (`0` = ข้อ 10) (เลือกได้หลายอย่าง) |
| `Space` หรือ `Enter` | บันทึก แล้วไปคลิปถัดไป |
| `B` | ย้อนกลับคลิปก่อนหน้า |
| `C` | ล้าง label ของคลิปนี้ |
| `Q` | บันทึกแล้วออก |

In [ ]:
import cv2
import json
import mediapipe as mp
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

INCORRECT_DIR = Path("dataset/incorrect")
LABELS_PATH = INCORRECT_DIR / "labels.json"
SEQUENCE_LENGTH = 60
FEATURES = 218
ONLY_UNLABELED = False        # True = ข้ามคลิปที่ label แล้ว
PLAYBACK_FPS = 10             # ความเร็วเล่นซ้ำ (10 = เท่าความเร็วจริง)

# รายการความผิด: code → (ชื่อภาษาไทย, วิธีทำท่าผิดตอนอัด)
ERROR_INFO = {
    "wrong_hand": (
        "มือผิดข้าง",
        "ทำวันทยหัตถ์ด้วย มือซ้าย แทนมือขวา"
    ),
    "hand_low": (
        "ปลายนิ้วต่ำกว่าคิ้ว",
        "ยกมือทำความเคารพ แต่ให้ปลายนิ้ว อยู่ระดับตา หรือแก้ม"
    ),
    "hand_high": (
        "ปลายนิ้วสูงกว่าคิ้ว",
        "ยกมือทำความเคารพ แต่ให้ปลายนิ้ว อยู่เหนือหน้าผาก"
    ),
    "elbow_at_shoulder": (
        "ศอกอยู่ระดับเดียวกับไหล่",
        "ยกมือทำความเคารพ แต่ยกศอกสูง จนอยู่ระดับเดียวกับไหล่"
    ),
    "body_tilted": (
        "ตัวเอียง",
        "ทำท่าวันทยหัตถ์ แต่เอียงตัว หรือเอียงไหล่"
    ),
    "heels_apart": (
        "ส้นเท้าไม่ชิด",
        "ทำท่าวันทยหัตถ์ แต่ยืนแยกส้นเท้า ออกจากกัน"
    ),
    "no_salute": (
        "ไม่ทำท่า",
        "ยืนตรงเฉย ๆ หรือทำท่าอื่น ไม่ยกมือทำความเคารพ"
    ),
    "no_head_turn": (
        "ไม่สะบัดหน้าไปทางซ้าย",
        "ยกอกอึ้บ ทำท่าครบ แต่หน้ามองตรง ไม่สะบัดหน้าไปทางซ้าย"
    ),
    "wrong_head_turn": (
        "สะบัดหน้าผิดไปทางขวา",
        "ทำท่าครบ แต่สะบัดหน้าไป ทางขวา แทนทางซ้าย"
    ),
    "left_arm_not_at_side": (
        "มือซ้ายไม่แนบลำตัว",
        "ทำท่าวันทยหัตถ์ แต่มือซ้าย กางออก ไม่แนบข้างลำตัว"
    ),
}
ERROR_TYPES = list(ERROR_INFO)
KEYS = "1234567890"            # ปุ่มเลือก label (ข้อ 10 = ปุ่ม 0)


# ตำแหน่งข้อมูลใน vector 218 ค่า (ต้องตรงกับ extract_landmarks ใน cell 1)
POSE_INDEXES = list(range(0, 11)) + [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
HAND_POINTS = 21
HAND_START = len(POSE_INDEXES) * 4

mp_holistic = mp.solutions.holistic
POSE_CONNECTIONS = [
    (a, b) for a, b in mp_holistic.POSE_CONNECTIONS
    if a in POSE_INDEXES and b in POSE_INDEXES
]
HAND_CONNECTIONS = list(mp_holistic.HAND_CONNECTIONS)

CANVAS_W, CANVAS_H = 640, 660
CENTER = (CANVAS_W // 2, 130)       # กึ่งกลางไหล่บนจอ
SCALE = 55                          # 1 ช่วงไหล่ = 55 px
WINDOW_NAME = "SaluteAI - Label incorrect clips"


def put_text(image, text, org, scale=0.6, color=(255, 255, 255), thickness=2):
    thickness += 1
    cv2.putText(image, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), thickness + 4, cv2.LINE_AA)
    cv2.putText(image, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)


# -------------------------
# Thai text (cv2 เขียนภาษาไทยไม่ได้ ใช้ Pillow + ฟอนต์ไทยแทน)
# -------------------------
THAI_FONT_CANDIDATES = [
    Path("../../KanitFont/Kanit-Bold.ttf"),
    Path("C:/Windows/Fonts/Kanit-Medium.ttf"),
    Path("C:/Windows/Fonts/tahomabd.ttf"),
    Path("C:/Windows/Fonts/LeelawUI.ttf"),
]
_thai_fonts = {}


def thai_font(size):
    if size not in _thai_fonts:
        path = next((p for p in THAI_FONT_CANDIDATES if p.exists()), None)

        if path is None:
            raise FileNotFoundError("ไม่พบฟอนต์ภาษาไทย")

        _thai_fonts[size] = ImageFont.truetype(str(path), size)

    return _thai_fonts[size]


def wrap_thai(draw, text, font, max_width):
    # ตัดบรรทัดตรงช่องว่าง เมื่อข้อความยาวเกินจอ
    lines, line = [], ""

    for word in text.split(" "):
        candidate = f"{line} {word}".strip()

        if line and draw.textlength(candidate, font=font) > max_width:
            lines.append(line)
            line = word
        else:
            line = candidate

    lines.append(line)
    return lines


def draw_thai(image, items):
    # items: [(text, (x, y), size, (B, G, R)), ...] ตัวหนังสือขอบดำ หนา อ่านชัด
    pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)

    for text, (x, y), size, color in items:
        font = thai_font(size)

        for line in wrap_thai(draw, text, font, image.shape[1] - x - 15):
            draw.text(
                (x, y), line, font=font,
                fill=tuple(color[::-1]),
                stroke_width=max(2, size // 10),
                stroke_fill=(0, 0, 0)
            )
            y += int(size * 1.35)

    image[:] = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


def to_pixel(x, y):
    return (
        int(CENTER[0] + x * SCALE),
        int(CENTER[1] + y * SCALE)
    )


def draw_skeleton(canvas, frame):
    # Pose
    pose = frame[:HAND_START].reshape(-1, 4)
    points = {
        index: to_pixel(pose[k, 0], pose[k, 1])
        for k, index in enumerate(POSE_INDEXES)
    }

    for a, b in POSE_CONNECTIONS:
        cv2.line(canvas, points[a], points[b], (200, 200, 200), 2)

    for point in points.values():
        cv2.circle(canvas, point, 4, (0, 140, 255), -1)

    # Hands (ซ้าย = ฟ้า, ขวา = ชมพู) ถ้าเป็น 0 ทั้งหมด = ตรวจไม่เจอมือ
    hands = frame[HAND_START:].reshape(2, HAND_POINTS, 3)

    for hand, color in zip(hands, [(255, 180, 0), (180, 0, 255)]):
        if not np.any(hand):
            continue

        hand_points = [to_pixel(x, y) for x, y, _ in hand]

        for a, b in HAND_CONNECTIONS:
            cv2.line(canvas, hand_points[a], hand_points[b], color, 2)


def load_clip(folder):
    return np.stack([
        np.load(folder / f"{i}.npy")
        for i in range(SEQUENCE_LENGTH)
    ])


labels = {}

if LABELS_PATH.exists():
    labels = json.loads(LABELS_PATH.read_text(encoding="utf-8"))

folders = sorted(
    (d for d in INCORRECT_DIR.iterdir() if d.is_dir() and d.name.isdigit()),
    key=lambda d: int(d.name)
)

if ONLY_UNLABELED:
    folders = [d for d in folders if not labels.get(d.name)]

if not folders:
    raise RuntimeError("ไม่มีคลิปให้ label")


def save_labels():
    LABELS_PATH.write_text(
        json.dumps(
            dict(sorted(labels.items(), key=lambda item: int(item[0]))),
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


position = 0
frame_num = 0
clip = load_clip(folders[position])
selected = set(labels.get(folders[position].name, []))

try:
    while True:
        folder = folders[position]

        canvas = np.full((CANVAS_H, CANVAS_W, 3), 30, dtype=np.uint8)
        draw_skeleton(canvas, clip[frame_num])

        put_text(canvas, f"Clip {folder.name}  ({position + 1}/{len(folders)})", (10, 25), 0.7)
        put_text(canvas, f"Frame {frame_num + 1}/{SEQUENCE_LENGTH}", (CANVAS_W - 170, 25), 0.5)

        # รายการ error ด้านล่าง: เขียว = เลือกแล้ว
        draw_thai(canvas, [
            (
                f"{KEYS[i]}: {ERROR_INFO[error][0]}",
                (10 + (i % 2) * 315, CANVAS_H - 182 + (i // 2) * 28),
                19,
                (0, 255, 0) if error in selected else (170, 170, 170)
            )
            for i, error in enumerate(ERROR_TYPES)
        ])

        put_text(canvas, "SPACE: next  B: back  C: clear  Q: quit", (10, CANVAS_H - 12), 0.5)

        cv2.imshow(WINDOW_NAME, canvas)

        key = cv2.waitKey(int(1000 / PLAYBACK_FPS)) & 0xFF
        frame_num = (frame_num + 1) % SEQUENCE_LENGTH

        if key == 255:
            continue

        if chr(key) in KEYS[:len(ERROR_TYPES)]:
            error = ERROR_TYPES[KEYS.index(chr(key))]
            selected ^= {error}

        elif key == ord("c"):
            selected.clear()

        elif key in (ord(" "), 13, ord("b"), ord("q")):
            labels[folder.name] = [e for e in ERROR_TYPES if e in selected]
            save_labels()

            if key == ord("q"):
                break

            step = -1 if key == ord("b") else 1

            if not 0 <= position + step < len(folders):
                if step == 1:
                    print("Label ครบทุกคลิปแล้ว")
                    break
                continue

            position += step
            frame_num = 0
            clip = load_clip(folders[position])
            selected = set(labels.get(folders[position].name, []))

finally:
    cv2.destroyAllWindows()

# สรุปจำนวนแต่ละประเภท
counts = {error: 0 for error in ERROR_TYPES}
unlabeled = []

for folder in sorted(
    (d for d in INCORRECT_DIR.iterdir() if d.is_dir() and d.name.isdigit()),
    key=lambda d: int(d.name)
):
    errors = labels.get(folder.name, [])

    if not errors:
        unlabeled.append(folder.name)

    for error in errors:
        counts[error] += 1

print("\nLabel summary:")

for error, count in counts.items():
    print(f"  {error:15s} {count}")

print("Unlabeled clips:", unlabeled if unlabeled else "none")

# 2. สร้าง datasets ที่ถูกและผิด
ทุกคลิปจะถูกตัดให้ **เริ่มตอนเริ่มยกมือ** (ก่อนยกมือ 0.3 วิ ยาว 4 วิ) ก่อนสร้างคู่ — Siamese จึงไม่สนว่ายกมือที่วินาทีไหน

Siamese ใช้ 111 ค่า/เฟรม (จาก 218): **ไม่ใช้ z (ความลึก)** เพราะ MediaPipe เดา z ไม่นิ่ง และ **ไม่ใช้จุดมืออีกข้าง** (ท่าถูกใน dataset ไม่เคยจับมือข้างนี้ได้ Siamese เลยเรียนผิดว่า "เห็นมืออีกข้าง = ผิด") — ดู `SIAMESE_COLUMNS` ใน `salute_rules.py`

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split
import importlib

import salute_rules as rules_lib
importlib.reload(rules_lib)

DATA_DIR = Path("dataset")
OUTPUT_DIR = Path("prepared")
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)


SEQUENCE_LENGTH = 60   # 6 วินาที x 10 FPS
FEATURES = 218   # 23 pose joints x 4 + 2 hands x 21 x 3


def load_sequence(path):
    # Folder แบบ LSTM-learn: dataset/<class>/<n>/0.npy ... 29.npy
    if path.is_dir():
        frames = [
            np.load(path / f"{i}.npy")
            for i in range(SEQUENCE_LENGTH)
        ]
        return np.stack(frames).astype(np.float32)

    # ไฟล์เดี่ยวแบบเดิม: dataset/<class>/<name>.npy (SEQUENCE_LENGTH, FEATURES)
    return np.load(path).astype(np.float32)


def sequence_paths(class_dir):
    folders = sorted(
        (d for d in class_dir.iterdir() if d.is_dir() and d.name.isdigit()),
        key=lambda d: int(d.name)
    )
    return folders + sorted(class_dir.glob("*.npy"))


def load_class(class_name):
    sequences = []

    for file in sequence_paths(DATA_DIR / class_name):
        try:
            sequence = load_sequence(file)
        except (FileNotFoundError, ValueError) as error:
            print("Skipping incomplete clip:", file, error)
            continue

        if sequence.shape != (SEQUENCE_LENGTH, FEATURES):
            print("Skipping invalid shape:", file, sequence.shape)
            continue

        if not np.isfinite(sequence).all():
            print("Skipping invalid values:", file)
            continue

        sequences.append(sequence)

    if len(sequences) < 5:
        raise ValueError(
            f"Need at least 5 valid sequences in {class_name}. "
            "Collect more data first."
        )

    return np.stack(sequences)


correct = load_class("correct")
incorrect = load_class("incorrect")

# เริ่มนับตอนเริ่มยกมือ: ตัดทุกคลิปให้เริ่มก่อนยกมือ 0.3 วิ ยาว 4 วิ (40 เฟรม)
# เพราะแต่ละคลิปยกมือไม่พร้อมกัน (บางคลิปวินาทีที่ 1.5 บางคลิป 3.0)
aligned_correct = [
    rules_lib.siamese_input(s, fallback_to_start=False) for s in correct
]
skipped = sum(a is None for a in aligned_correct)

if skipped:
    print(f"ท่าถูก {skipped} คลิปหาจุดเริ่มยกมือไม่เจอ (มือยกค้างตั้งแต่ต้นคลิป) - ไม่ใช้")

correct = np.stack([a for a in aligned_correct if a is not None])

# ท่าผิดที่ไม่ได้ยกมือ (เช่น ไม่ทำท่า) ใช้ต้นคลิปแทน
# siamese_input = ตัดตามจุดเริ่มยกมือ + ตัดค่า z (ความลึก) ออก
incorrect = np.stack([rules_lib.siamese_input(s) for s in incorrect])

print("Correct (aligned):", correct.shape)
print("Incorrect (aligned):", incorrect.shape)


# Split each class separately before creating pairs
def split_sequences(data):
    train, temp = train_test_split(
        data,
        test_size=0.30,
        random_state=SEED
    )

    val, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=SEED
    )

    return train, val, test


correct_train, correct_temp_val, correct_temp_test = split_sequences(correct)
incorrect_train, incorrect_temp_val, incorrect_temp_test = split_sequences(incorrect)


# Create balanced positive and negative pairs
def make_pairs(correct_data, incorrect_data):
    if len(correct_data) < 2 or len(incorrect_data) < 2:
        raise ValueError("Each class needs at least 2 sequences per split.")

    pair_a = []
    pair_b = []
    labels = []

    number_of_pairs = 2 * max(
        len(correct_data),
        len(incorrect_data)
    )

    for _ in range(number_of_pairs):
        # Positive pair: two correct examples
        i, j = rng.choice(
            len(correct_data), size=2, replace=False
        )

        pair_a.append(correct_data[i])
        pair_b.append(correct_data[j])
        labels.append(1.0)

        # Negative pair: one correct and one incorrect
        i = rng.integers(len(correct_data))
        j = rng.integers(len(incorrect_data))

        pair_a.append(correct_data[i])
        pair_b.append(incorrect_data[j])
        labels.append(0.0)

    pair_a = np.stack(pair_a).astype(np.float32)
    pair_b = np.stack(pair_b).astype(np.float32)
    labels = np.array(labels, dtype=np.float32)

    # Shuffle the pairs
    order = rng.permutation(len(labels))

    return pair_a[order], pair_b[order], labels[order]


splits = {
    "train": (correct_train, incorrect_train),
    "val": (correct_temp_val, incorrect_temp_val),
    "test": (correct_temp_test, incorrect_temp_test)
}

for split_name, (c_data, i_data) in splits.items():
    a, b, y = make_pairs(c_data, i_data)

    output_path = OUTPUT_DIR / f"{split_name}.npz"

    np.savez_compressed(
        output_path,
        pair_a=a,
        pair_b=b,
        labels=y
    )

    print(
        f"{split_name}:",
        a.shape,
        b.shape,
        y.shape,
        "saved to", output_path
    )

# 3. Build Siamese

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

Path("models").mkdir(exist_ok=True)

SEED = 42
tf.keras.utils.set_random_seed(SEED)

SEQUENCE_LENGTH = 40   # aligned: 4 วินาที นับจากเริ่มยกมือ (salute_rules.ALIGN_LENGTH)
FEATURES = 111   # ไม่ใช้ z + ไม่ใช้จุดมืออีกข้าง (salute_rules.SIAMESE_FEATURES)


# -------------------------
# 1. Load prepared pairs
# -------------------------
def load_pairs(path):
    data = np.load(path)

    return (
        data["pair_a"].astype(np.float32),
        data["pair_b"].astype(np.float32),
        data["labels"].astype(np.float32)
    )


a_train, b_train, y_train = load_pairs("prepared/train.npz")
a_val, b_val, y_val = load_pairs("prepared/val.npz")


# -------------------------
# 2. Create shared encoder
# -------------------------
def create_encoder():
    inputs = layers.Input(
        shape=(SEQUENCE_LENGTH, FEATURES)
    )

    x = layers.LSTM(128)(inputs)
    x = layers.Dense(64, activation="relu")(x)

    # Embedding vector
    outputs = layers.Dense(32)(x)

    return Model(
        inputs=inputs,
        outputs=outputs,
        name="shared_encoder"
    )


encoder = create_encoder()


# -------------------------
# 3. Siamese inputs
# -------------------------
input_a = layers.Input(
    shape=(SEQUENCE_LENGTH, FEATURES),
    name="sequence_a"
)

input_b = layers.Input(
    shape=(SEQUENCE_LENGTH, FEATURES),
    name="sequence_b"
)

embedding_a = encoder(input_a)
embedding_b = encoder(input_b)


# -------------------------
# 4. Euclidean distance
# -------------------------
def euclidean_distance(tensors):
    a, b = tensors

    squared_distance = tf.reduce_sum(
        tf.square(a - b),
        axis=1,
        keepdims=True
    )

    return tf.sqrt(
        tf.maximum(squared_distance, 1e-10)
    )


distance = layers.Lambda(
    euclidean_distance,
    name="distance"
)([embedding_a, embedding_b])


siamese = Model(
    inputs=[input_a, input_b],
    outputs=distance,
    name="siamese_network"
)


# -------------------------
# 5. Contrastive loss
# -------------------------
def contrastive_loss(y_true, y_pred):
    y_true = tf.cast(
        tf.reshape(y_true, (-1, 1)),
        tf.float32
    )

    margin = 1.0

    similar_loss = y_true * tf.square(y_pred)

    different_loss = (
        (1.0 - y_true)
        * tf.square(
            tf.maximum(margin - y_pred, 0.0)
        )
    )

    return tf.reduce_mean(
        similar_loss + different_loss
    )


# -------------------------
# 6. Compile
# -------------------------
siamese.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=contrastive_loss
)

siamese.summary()


# -------------------------
# 7. Train
# -------------------------
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/best_siamese.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history = siamese.fit(
    [a_train, b_train],
    y_train,
    validation_data=(
        [a_val, b_val],
        y_val
    ),
    epochs=50,
    batch_size=16,
    callbacks=callbacks
)

# Save the shared encoder for later comparison
encoder.save("models/encoder.keras")

print("Training complete!")

# 4. Eveluate Model

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# โหลดแค่ encoder (ไม่มี Lambda layer) แล้วคำนวณระยะห่างเอง
# เพราะ best_siamese.keras มี Lambda layer ที่โหลดกลับไม่ได้ใน Keras 2.15
ENCODER_PATH = "models/encoder.keras"

encoder = tf.keras.models.load_model(
    ENCODER_PATH,
    compile=False
)


def pair_distances(a, b):
    embedding_a = encoder.predict(a, verbose=0)
    embedding_b = encoder.predict(b, verbose=0)

    return np.linalg.norm(
        embedding_a - embedding_b,
        axis=1
    )


def load_pairs(path):
    data = np.load(path)

    return (
        data["pair_a"].astype(np.float32),
        data["pair_b"].astype(np.float32),
        data["labels"].astype(np.int32)
    )


# Get distances for each pair
def get_distances(path):
    a, b, y = load_pairs(path)

    distances = pair_distances(a, b)

    return distances, y


# -------------------------
# 1. Find threshold on VAL
# -------------------------
val_distances, y_val = get_distances(
    "prepared/val.npz"
)

candidates = np.linspace(
    val_distances.min(),
    val_distances.max(),
    500
)

accuracies = np.array([
    accuracy_score(y_val, (val_distances <= threshold).astype(int))
    for threshold in candidates
])

# ช่วง threshold ที่ได้ accuracy สูงสุด → เลือกค่ากลาง
# (ค่าต่ำสุดของช่วงจะแคบเกินไป ท่าถูกที่ต่างจาก dataset นิดเดียวก็ไม่ผ่าน)
best_accuracy = float(accuracies.max())
best_range = candidates[np.isclose(accuracies, best_accuracy)]
best_threshold = float((best_range.min() + best_range.max()) / 2)


Path("models/threshold.txt").write_text(
    str(best_threshold)
)

print("Chosen threshold:", best_threshold)
print("Validation pair accuracy:", best_accuracy)


# -------------------------
# 2. Evaluate on TEST once
# -------------------------
test_distances, y_test = get_distances(
    "prepared/test.npz"
)

test_predictions = (
    test_distances <= best_threshold
).astype(int)

print("\nTEST RESULTS")
print("Accuracy:", accuracy_score(
    y_test, test_predictions
))
print("Precision:", precision_score(
    y_test, test_predictions, zero_division=0
))
print("Recall:", recall_score(
    y_test, test_predictions, zero_division=0
))
print("F1:", f1_score(
    y_test, test_predictions, zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test, test_predictions
))


## 4.5 Calibrate feedback rules
หาเกณฑ์ของกฎ feedback แต่ละข้อจาก dataset (ท่าถูก + ท่าผิดที่ติด label) แล้วบันทึกเป็น `models/rules.json`
- กฎทั้งหมดอยู่ในไฟล์ `salute_rules.py` (ใช้ร่วมกับ cell 5 Test)
- **รันใหม่ทุกครั้งที่อัด dataset เพิ่ม** (ไม่ต้องเทรน Siamese ใหม่ก็ได้ ถ้าแค่ต้องการปรับกฎ)

In [ ]:
import json
import importlib
from pathlib import Path

import salute_rules as rules_lib
importlib.reload(rules_lib)       # โหลดใหม่ถ้ามีการแก้ salute_rules.py

DATA_DIR = Path("dataset")
SEQUENCE_LENGTH = 60
RULES_PATH = Path("models/rules.json")
LABELS_PATH = DATA_DIR / "incorrect" / "labels.json"


def clip_folders(class_dir):
    return sorted(
        (d for d in class_dir.iterdir() if d.is_dir() and d.name.isdigit()),
        key=lambda d: int(d.name)
    )


def safe_load(folder):
    try:
        return rules_lib.load_clip(folder, SEQUENCE_LENGTH)
    except (FileNotFoundError, ValueError):
        print("Skipping broken clip:", folder)
        return None


correct = [
    clip for clip in (safe_load(d) for d in clip_folders(DATA_DIR / "correct"))
    if clip is not None
]

labels = {}

if LABELS_PATH.exists():
    labels = json.loads(LABELS_PATH.read_text(encoding="utf-8"))

labeled = []

for folder in clip_folders(DATA_DIR / "incorrect"):
    errors = labels.get(folder.name, [])
    clip = safe_load(folder) if errors else None

    if clip is not None:
        labeled.append((clip, errors))

print(f"ท่าถูก {len(correct)} คลิป | ท่าผิดที่มี label {len(labeled)} คลิป")

rules = rules_lib.calibrate(correct, labeled)
rules_lib.save_rules(rules, RULES_PATH)

print("Saved:", RULES_PATH)
print("มือที่ทำความเคารพ = pose", rules["side"]["wrist"])

# ผลตรวจกฎกับ dataset (ข้อมูลชุดเดียวกับที่ใช้หาเกณฑ์ จึงดูดีกว่าใช้จริงเล็กน้อย)
report = rules_lib.evaluate(correct, labeled, rules)

print(f"\n{'กฎ':28s}{'จับได้':>10s}{'เตือนผิด':>12s}")

for code in rules_lib.PRIORITY:
    name = rules_lib.FEEDBACK_TH[code][0]
    r = report[code]
    detected = f"{r['detected']}/{r['labeled']}" if r["labeled"] else "-"
    print(f"{name:28s}{detected:>10s}{r['false_alarms']:>12d}")

print(
    f"\nท่าถูกที่ไม่โดนเตือนเลย: "
    f"{report['_correct_clean']}/{report['_correct_total']} คลิป"
)

# 5. Test

In [29]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from pathlib import Path
from collections import deque
import time
import importlib
from PIL import Image, ImageDraw, ImageFont

import salute_rules as rules_lib
importlib.reload(rules_lib)

# -------------------------
# 1. Configuration
# -------------------------
CLIP_SECONDS = 6              # ความยาวแต่ละคลิป (วินาที)
SAMPLE_FPS = 10               # เก็บ 10 เฟรมต่อวินาที
SEQUENCE_LENGTH = CLIP_SECONDS * SAMPLE_FPS   # = 60 เฟรมต่อคลิป
PREDICT_INTERVAL = 0.5        # ทำนายทุก 0.5 วินาที (ลดภาระเครื่อง)
FEATURES = 218   # 23 pose joints x 4 + 2 hands x 21 x 3
CAMERA_ID = 0

CORRECT_DIR = Path("dataset/correct")
ENCODER_PATH = "models/encoder.keras"
THRESHOLD_PATH = Path("models/threshold.txt")
RULES_PATH = Path("models/rules.json")
MAX_FEEDBACK = 3              # แสดง feedback พร้อมกันสูงสุดกี่ข้อ

if not RULES_PATH.exists():
    raise FileNotFoundError("ไม่พบ models/rules.json - รัน cell 4.5 Calibrate ก่อน")

feedback_rules = rules_lib.load_rules(RULES_PATH)

if not THRESHOLD_PATH.exists():
    raise FileNotFoundError(
        "Run evaluate_model.py first."
    )

threshold = float(
    THRESHOLD_PATH.read_text().strip()
)

if threshold <= 0:
    raise ValueError(
        "Threshold must be greater than zero."
    )

# Load trained shared encoder (ไม่ใช้ best_siamese.keras เพราะมี Lambda layer)
encoder = tf.keras.models.load_model(
    ENCODER_PATH,
    compile=False
)

# Load correct reference sequences
# รองรับทั้งแบบโฟลเดอร์ (LSTM-learn) และไฟล์ .npy เดี่ยวแบบเดิม
reference_paths = sorted(
    (d for d in CORRECT_DIR.iterdir() if d.is_dir() and d.name.isdigit()),
    key=lambda d: int(d.name)
) + sorted(CORRECT_DIR.glob("*.npy"))

references = []

for path in reference_paths:
    try:
        if path.is_dir():
            seq = np.stack([
                np.load(path / f"{i}.npy")
                for i in range(SEQUENCE_LENGTH)
            ]).astype(np.float32)
        else:
            seq = np.load(path).astype(np.float32)
    except (FileNotFoundError, ValueError):
        continue

    if seq.shape == (SEQUENCE_LENGTH, FEATURES):
        references.append(seq)

if not references:
    raise RuntimeError(
        "No valid correct reference sequences found."
    )

# จัดจังหวะให้เริ่มตอนยกมือ เหมือนตอนเทรน (ตัดคลิปที่หาจุดเริ่มยกมือไม่เจอออก)
references = [rules_lib.siamese_input(s, fallback_to_start=False) for s in references]
references = np.stack([s for s in references if s is not None])

# คำนวณ embedding ของท่าต้นแบบไว้ครั้งเดียว
reference_embeddings = encoder.predict(
    references,
    verbose=0
)

print("Reference sequences:", len(references))
print("Threshold:", threshold)


# -------------------------
# 2. MediaPipe setup
# -------------------------
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils


# -------------------------
# 2. Extract 218 features
# -------------------------
# Pose joints ที่ใช้ (index ของ MediaPipe Pose)
POSE_JOINTS = {
    "face": list(range(0, 11)),   # จมูก ตา หู ปาก (11 จุด)
    "shoulders": [11, 12],
    "elbows": [13, 14],
    "wrists": [15, 16],
    "hips": [23, 24],             # สะโพก (ต้นขา)
    "knees": [25, 26],            # เข่า
    "ankles": [27, 28],           # ข้อเท้า
}
POSE_INDEXES = [i for group in POSE_JOINTS.values() for i in group]

HAND_POINTS = 21                  # จุดบนมือข้างละ 21 จุด (นิ้วครบ)

# 23 pose x (x, y, z, visibility) + 2 มือ x 21 x (x, y, z) = 92 + 126 = 218
FEATURES = len(POSE_INDEXES) * 4 + 2 * HAND_POINTS * 3


def extract_landmarks(results):
    if results.pose_landmarks is None:
        return None

    landmarks = results.pose_landmarks.landmark

    # Shoulder landmarks
    left_shoulder = landmarks[11]
    right_shoulder = landmarks[12]

    center_x = (
        left_shoulder.x + right_shoulder.x
    ) / 2

    center_y = (
        left_shoulder.y + right_shoulder.y
    ) / 2

    scale = np.sqrt(
        (left_shoulder.x - right_shoulder.x) ** 2
        + (left_shoulder.y - right_shoulder.y) ** 2
    )

    scale = max(scale, 1e-6)

    values = []

    # Pose: หน้า ไหล่ ศอก ข้อมือ สะโพก เข่า ข้อเท้า
    for index in POSE_INDEXES:
        landmark = landmarks[index]

        # Normalize coordinates relative to shoulders
        x = (landmark.x - center_x) / scale
        y = (landmark.y - center_y) / scale
        z = landmark.z / scale
        visibility = landmark.visibility

        values.extend([x, y, z, visibility])

    # Hands: ถ้าตรวจไม่เจอมือข้างไหน ใส่ 0 แทน (เหมือน LSTM-learn)
    for hand in [
        results.left_hand_landmarks,
        results.right_hand_landmarks
    ]:
        if hand is None:
            values.extend([0.0] * HAND_POINTS * 3)
            continue

        for landmark in hand.landmark:
            x = (landmark.x - center_x) / scale
            y = (landmark.y - center_y) / scale
            z = landmark.z / scale

            values.extend([x, y, z])

    return np.array(values, dtype=np.float32)


def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_holistic.POSE_CONNECTIONS
    )
    mp_drawing.draw_landmarks(
        image,
        results.left_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS
    )
    mp_drawing.draw_landmarks(
        image,
        results.right_hand_landmarks,
        mp_holistic.HAND_CONNECTIONS
    )


# -------------------------
# 3. Webcam
# -------------------------
def put_text(image, text, org, font, scale, color, thickness=2, line_type=cv2.LINE_AA):
    # ตัวหนังสือหนา มีขอบดำ อ่านชัดทุกพื้นหลัง
    thickness += 1
    cv2.putText(image, text, org, font, scale, (0, 0, 0), thickness + 4, line_type)
    cv2.putText(image, text, org, font, scale, color, thickness, line_type)


# -------------------------
# Thai text (cv2 เขียนภาษาไทยไม่ได้ ใช้ Pillow + ฟอนต์ไทยแทน)
# -------------------------
THAI_FONT_CANDIDATES = [
    Path("../../KanitFont/Kanit-Bold.ttf"),
    Path("C:/Windows/Fonts/Kanit-Medium.ttf"),
    Path("C:/Windows/Fonts/tahomabd.ttf"),
    Path("C:/Windows/Fonts/LeelawUI.ttf"),
]
_thai_fonts = {}


def thai_font(size):
    if size not in _thai_fonts:
        path = next((p for p in THAI_FONT_CANDIDATES if p.exists()), None)

        if path is None:
            raise FileNotFoundError("ไม่พบฟอนต์ภาษาไทย")

        _thai_fonts[size] = ImageFont.truetype(str(path), size)

    return _thai_fonts[size]


def draw_thai(image, items):
    # items: [(text, (x, y), size, (B, G, R)), ...] ตัวหนังสือขอบดำ หนา อ่านชัด
    pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)

    for text, (x, y), size, color in items:
        draw.text(
            (x, y), text, font=thai_font(size),
            fill=tuple(color[::-1]),
            stroke_width=max(2, size // 10),
            stroke_fill=(0, 0, 0)
        )

    image[:] = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


def darken(image, alpha, top=0, bottom=None):
    bottom = image.shape[0] if bottom is None else bottom
    image[top:bottom] = (image[top:bottom] * (1 - alpha)).astype(np.uint8)


def draw_feedback(image, errors, is_similar):
    # กล่อง feedback ภาษาไทยด้านล่างจอ
    height = image.shape[0]
    bottom = height - 40

    if errors:
        shown = errors[:MAX_FEEDBACK]
        top = bottom - 45 - 62 * len(shown)
        darken(image, 0.6, top, bottom)

        items = [("ต้องแก้ไข", (15, top + 5), 28, (80, 80, 255))]
        y = top + 45

        for code in shown:
            name, tip = rules_lib.FEEDBACK_TH[code]
            items.append((f"- {name}", (15, y), 24, (0, 165, 255)))
            items.append((tip, (35, y + 30), 20, (255, 255, 255)))
            y += 62

    elif is_similar:
        top = bottom - 60
        darken(image, 0.6, top, bottom)
        items = [("ท่าถูกต้อง ดีมาก!", (15, top + 10), 34, (0, 230, 0))]

    else:
        top = bottom - 80
        darken(image, 0.6, top, bottom)
        items = [
            ("ท่ายังไม่ตรงกับต้นแบบ", (15, top + 5), 28, (0, 165, 255)),
            ("ลองทำใหม่อีกครั้ง ให้ครบจังหวะภายใน 6 วินาที", (15, top + 45), 20, (255, 255, 255)),
        ]

    draw_thai(image, items)


def resample_sequence(times, frames):
    # เลือกเฟรมที่ใกล้เวลาเป้าหมายที่สุด ให้ได้ SEQUENCE_LENGTH เฟรมเท่ากันทุกคลิป
    # ไม่ว่ากล้อง + MediaPipe จะได้กี่ FPS
    times = np.asarray(times, dtype=np.float64)
    times = times - times[0]
    targets = np.linspace(0.0, times[-1], SEQUENCE_LENGTH)
    indexes = np.abs(times[None, :] - targets[:, None]).argmin(axis=1)

    return [frames[i] for i in indexes]


cap = cv2.VideoCapture(CAMERA_ID)

if not cap.isOpened():
    raise RuntimeError("Cannot open webcam.")

# (time, features) ย้อนหลัง CLIP_SECONDS วินาที
sequence_buffer = deque()
last_predict = 0.0
last_result = None
error_history = deque(maxlen=3)   # ผลกฎ 3 ครั้งล่าสุด (กันข้อความกระพริบ)

# Smooth the displayed score
score_buffer = deque(maxlen=5)

with mp_holistic.Holistic(
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as pose:

    while True:
        success, frame = cap.read()

        if not success:
            print("Cannot read webcam.")
            break

        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        results = pose.process(rgb)

        features = extract_landmarks(results)

        draw_landmarks(frame, results)

        if features is None:
            sequence_buffer.clear()
            score_buffer.clear()
            last_result = None
            error_history.clear()

            put_text(
                frame,
                "Body not detected",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (0, 0, 255), 2
            )

        else:
            now = time.time()
            sequence_buffer.append((now, features))

            # เก็บเฉพาะเฟรมย้อนหลัง CLIP_SECONDS วินาที
            while now - sequence_buffer[0][0] > CLIP_SECONDS:
                sequence_buffer.popleft()

            buffered_seconds = now - sequence_buffer[0][0]

            put_text(
                frame,
                f"Buffer: {buffered_seconds:.1f}/{CLIP_SECONDS}s",
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 255, 255), 2
            )

            buffer_ready = buffered_seconds >= CLIP_SECONDS - 0.3

            if buffer_ready and now - last_predict >= PREDICT_INTERVAL:
                last_predict = now

                buffer_times, buffer_frames = zip(*sequence_buffer)

                live_sequence = np.array(
                    resample_sequence(buffer_times, buffer_frames),
                    dtype=np.float32
                )

                # Compare live embedding with each reference embedding
                # ตัดให้เริ่มตอนยกมือ (หาไม่เจอ = ใช้ต้นช่วง 6 วิ)
                live_aligned = rules_lib.siamese_input(live_sequence)

                live_embedding = encoder.predict(
                    live_aligned[np.newaxis, ...],
                    verbose=0
                )

                distances = np.linalg.norm(
                    reference_embeddings - live_embedding,
                    axis=1
                )

                # Use the mean of the 3 nearest references
                nearest_count = min(3, len(distances))

                nearest_distances = np.sort(
                    distances
                )[:nearest_count]

                distance = float(
                    np.mean(nearest_distances)
                )

                # Heuristic display score, NOT calibrated accuracy
                similarity = 100.0 * max(
                    0.0,
                    min(
                        1.0,
                        1.0 - distance / (2.0 * threshold)
                    )
                )

                score_buffer.append(similarity)

                displayed_score = float(
                    np.mean(score_buffer)
                )

                is_similar = distance <= threshold

                if is_similar:
                    status = "SIMILAR TO REFERENCE"
                    color = (0, 220, 0)
                else:
                    status = "CHECK YOUR POSE"
                    color = (0, 165, 255)

                # --- Feedback rules: ผิดตรงไหน ---
                error_history.append(
                    rules_lib.detect_errors(live_sequence, feedback_rules)
                )

                # แสดงเฉพาะข้อที่เจอซ้ำอย่างน้อย 2 ใน 3 ครั้งล่าสุด
                need = min(2, len(error_history))
                errors = [
                    code for code in rules_lib.PRIORITY
                    if sum(code in found for found in error_history) >= need
                ]

                if errors:
                    status = "FIX YOUR POSE"
                    color = (0, 0, 255)

                last_result = (
                    displayed_score, status, color, distance, errors, is_similar
                )

            if last_result is not None:
                displayed_score, status, color, distance, errors, is_similar = last_result

                put_text(
                    frame,
                    f"Similarity: {displayed_score:.1f}%",
                    (20, 75),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, color, 2
                )

                put_text(
                    frame,
                    status,
                    (20, 115),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65, color, 2
                )

                put_text(
                    frame,
                    f"Distance: {distance:.3f}",
                    (20, 150),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 255, 255), 2
                )

                draw_feedback(frame, errors, is_similar)

        put_text(
            frame,
            "Q: Quit",
            (20, frame.shape[0] - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (255, 255, 255), 2
        )

        cv2.imshow("SaluteAI - Realtime", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

Reference sequences: 29
Threshold: 0.5281257927231118
